# Data Exploration
This notebook performs exploratory data analysis (EDA) on the smartphone addiction dataset, visualizing distributions, handling missing values, and identifying key patterns.



This notebook performs an initial exploratory data analysis on the `train.csv` dataset, checking for missing values, basic statistics, and validating a few initial hypotheses.

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('../datasets/train.csv')

# Quick preview
df.head()

## Basic Information
Let's look at the shape of the data, column types, and missing values.

In [ ]:
print(f"Number of Rows: {df.shape[0]}")
print(f"Number of Columns: {df.shape[1]}")

print("\n--- Missing Values ---")
print(df.isnull().sum())

## Summary Statistics
Let's check the mean, min, and max for our numerical columns.

In [ ]:
df.describe().loc[['mean', 'min', 'max']].T

## Initial Hypotheses

Based on the initial features, we can formulate the following hypotheses to test in future analyses:

1. **Screen Time Hypothesis**: Higher `daily_screen_time_hours` and `weekend_screen_time` correlate positively with the `addicted_label`.
2. **App Usage Hypothesis**: Users with a high number of `app_opens_per_day` and `notifications_per_day` are more likely to have higher `stress_level`s and an `addicted_label`.
3. **Sleep Deprivation Hypothesis**: Higher `addicted_label` instances are associated with fewer `sleep_hours`.
4. **Work/Study Impact Hypothesis**: Users labeled as addicted will have a higher likelihood of reporting 'Yes' in `academic_work_impact`.
5. **Activity Specific Hypothesis**: `social_media_hours` and `gaming_hours` contribute more significantly to addiction than general `work_study_hours`.

In [ ]:
# Let's check the distribution of the target variable 'addicted_label'
print("Target Variable Distribution:")
print(df['addicted_label'].value_counts(normalize=True))

## 3. Data Cleaning & Imputation
Based on our strategy, we will impute categorical missing values with 'Unknown' and numerical missing values with the Median.

In [ ]:
# Identify numerical and categorical columns
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

# Exclude target and ID from imputation just in case
numerical_cols = [c for c in numerical_cols if c not in ['id', 'addicted_label']]

medians = df[numerical_cols].median()
components = ['social_media_hours', 'gaming_hours', 'work_study_hours']
missing_masks = {col: df[col].isna() for col in components}
missing_daily = df['daily_screen_time_hours'].isna()

# Impute Categorical with 'Unknown'
for col in categorical_cols:
    df[col] = df[col].fillna('Unknown')

# For non-screen time variables, impute with median
for col in numerical_cols:
    if col not in components and col != 'daily_screen_time_hours':
        df[col] = df[col].fillna(medians[col])

import numpy as np
C_orig = df[components].fillna(0).sum(axis=1)
C_imp_sum = sum((missing_masks[col].astype(int) * medians[col]) for col in components)
mask_daily_present = ~missing_daily
violation_mask = mask_daily_present & (C_orig + C_imp_sum > df['daily_screen_time_hours'])
T_avail = np.maximum(0, df['daily_screen_time_hours'] - C_orig)

for col in components:
    df[col] = df[col].fillna(medians[col])
    if missing_masks[col].any():
        prop = np.where(C_imp_sum > 0, medians[col] / C_imp_sum, 0)
        df.loc[violation_mask & missing_masks[col], col] = T_avail[violation_mask] * prop[violation_mask]

df['daily_screen_time_hours'] = df['daily_screen_time_hours'].fillna(medians['daily_screen_time_hours'])
total_components = df[components].sum(axis=1)
mask_missing_daily_violation = missing_daily & (total_components > df['daily_screen_time_hours'])
df.loc[mask_missing_daily_violation, 'daily_screen_time_hours'] = total_components[mask_missing_daily_violation]

print("\n--- Missing Values After Imputation ---")
print(df.isnull().sum())

## 4. Hypothesis Testing
Let's dive deep into testing the hypotheses outlined above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set plotting style
sns.set_theme(style="whitegrid")

### Hypothesis 1: Screen Time
Higher `daily_screen_time_hours` and `weekend_screen_time` correlate positively with the `addicted_label`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='addicted_label', y='daily_screen_time_hours', ax=axes[0])
axes[0].set_title('Daily Screen Time by Addiction Status')

sns.boxplot(data=df, x='addicted_label', y='weekend_screen_time', ax=axes[1])
axes[1].set_title('Weekend Screen Time by Addiction Status')

plt.show()

# Statistical Test (Mann-Whitney U)
addicted = df[df['addicted_label'] == 1]
not_addicted = df[df['addicted_label'] == 0]

stat, p = stats.mannwhitneyu(addicted['daily_screen_time_hours'], not_addicted['daily_screen_time_hours'], alternative='greater')
print(f"Daily Screen Time MWU p-value: {p}")

stat, p = stats.mannwhitneyu(addicted['weekend_screen_time'], not_addicted['weekend_screen_time'], alternative='greater')
print(f"Weekend Screen Time MWU p-value: {p}")

### Hypothesis 2: App Usage
Users with a high number of `app_opens_per_day` and `notifications_per_day` are more likely to have higher stress and addiction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='addicted_label', y='app_opens_per_day', ax=axes[0])
axes[0].set_title('App Opens by Addiction Status')

sns.boxplot(data=df, x='addicted_label', y='notifications_per_day', ax=axes[1])
axes[1].set_title('Notifications by Addiction Status')

plt.show()

# Test
stat, p = stats.mannwhitneyu(addicted['app_opens_per_day'], not_addicted['app_opens_per_day'], alternative='greater')
print(f"App Opens MWU p-value: {p}")

stat, p = stats.mannwhitneyu(addicted['notifications_per_day'], not_addicted['notifications_per_day'], alternative='greater')
print(f"Notifications MWU p-value: {p}")

### Hypothesis 3: Sleep Deprivation
Higher addicted label instances are associated with fewer `sleep_hours`.

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x='addicted_label', y='sleep_hours')
plt.title('Sleep Hours by Addiction Status')
plt.show()

# Test (alternative='less' because we expect addicted users to sleep less)
stat, p = stats.mannwhitneyu(addicted['sleep_hours'], not_addicted['sleep_hours'], alternative='less')
print(f"Sleep Hours MWU p-value: {p}")

### Hypothesis 4: Work/Study Impact
Users labeled as addicted will have a higher likelihood of reporting 'Yes' in `academic_work_impact`.

In [ ]:
# Crosstab
impact_crosstab = pd.crosstab(df['addicted_label'], df['academic_work_impact'])
print(impact_crosstab)

# Plot
impact_crosstab.plot(kind='bar', stacked=True, figsize=(8, 5))
plt.title('Academic/Work Impact by Addiction Status')
plt.show()

# Chi-Square Test
chi2, p, dof, ex = stats.chi2_contingency(impact_crosstab)
print(f"Chi-Square p-value: {p}")

### Hypothesis 5: Activity Specificity
`social_media_hours` and `gaming_hours` contribute more significantly to addiction than `work_study_hours`.

In [ ]:
activities = ['social_media_hours', 'gaming_hours', 'work_study_hours', 'addicted_label']
corr = df[activities].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation of Activity Types with Addiction')
plt.show()